# Notebook 05 - Modelo Preditivo: Risco de Cancelamento
**Projeto:** Análise de Custos e Margem por Categoria — Olist E-Commerce
**Autor:** Ariel Marquezin
**Stack:** Scikit-learn · pandas · numpy · Matplotlib · Seaborn

In [ ]:
# Instalacao (somente se necessario)
# !pip install scikit-learn pandas numpy matplotlib seaborn pyarrow -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)
import sklearn

print(f'Scikit-learn : {sklearn.__version__}')
print(f'pandas       : {pd.__version__}')
print('Imports OK')

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'axes.titlecolor': '#f0f6fc', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'text.color': '#c9d1d9',
    'grid.color': '#21262d', 'grid.alpha': 0.5,
    'font.family': 'monospace',
    'axes.spines.top': False, 'axes.spines.right': False,
})

C_CYAN   = '#00e5ff'
C_GREEN  = '#10b981'
C_YELLOW = '#f59e0b'
C_RED    = '#ef4444'
C_BLUE   = '#3b82f6'

GOLD_PATH    = '/content/drive/MyDrive/olist/gold/'
FIGURES_PATH = '/content/drive/MyDrive/olist/figures/'
os.makedirs(FIGURES_PATH, exist_ok=True)

In [ ]:
# Carregar dados
df_raw = pq.read_table(os.path.join(GOLD_PATH, 'metricas_pedidos.parquet')).to_pandas()
print(f'Dataset: {len(df_raw):,} linhas x {len(df_raw.columns)} colunas')
print(df_raw['order_status'].value_counts())

In [ ]:
# Feature Engineering
df = df_raw.copy()
df = df[df['order_status'].isin(['delivered', 'canceled'])].copy()
df['target'] = np.where(df['order_status'] == 'canceled', 1, 0)

df['purchase_date'] = pd.to_datetime(df['purchase_date'], errors='coerce')
df['purchase_hour']      = df['purchase_date'].dt.hour
df['purchase_dayofweek'] = df['purchase_date'].dt.dayofweek
df['purchase_month']     = df['purchase_date'].dt.month
df['is_weekend']         = np.where(df['purchase_dayofweek'] >= 5, 1, 0)
df['freight_ratio']      = df['freight_value'] / df['price'].replace(0, np.nan)
df['is_installment']     = np.where(df['payment_installments'] > 1, 1, 0)
df['high_ticket']        = np.where(df['price'] > df['price'].quantile(0.75), 1, 0)
df['payment_type']       = df['payment_type'].fillna('unknown')
df['category']           = df['category'].fillna('unknown')

FEATURES_NUM = [
    'price', 'freight_value', 'freight_ratio',
    'payment_installments', 'review_score',
    'purchase_hour', 'purchase_dayofweek', 'purchase_month',
    'is_weekend', 'is_installment', 'high_ticket'
]
FEATURES_CAT = ['category', 'payment_type']
TARGET = 'target'

df_model = df[FEATURES_NUM + FEATURES_CAT + [TARGET]].dropna()
print(f'Dataset de modelagem: {len(df_model):,} linhas')
print(f'Taxa de cancelamento: {df_model[TARGET].mean()*100:.2f}%')

In [ ]:
# Split e Pre-processamento
X = df_model[FEATURES_NUM + FEATURES_CAT]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino : {len(X_train):,} amostras')
print(f'Teste  : {len(X_test):,} amostras')

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), FEATURES_CAT),
])

In [ ]:
# Treinar 3 modelos
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=500, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=8,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=120, max_depth=4, learning_rate=0.08, random_state=42
    ),
}

pipelines = {}
results   = {}
cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    auc     = roc_auc_score(y_test, y_proba)
    ap      = average_precision_score(y_test, y_proba)
    cv_auc  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
    pipelines[name] = pipe
    results[name]   = {'pipeline': pipe, 'y_pred': y_pred, 'y_proba': y_proba,
                       'roc_auc': auc, 'avg_precision': ap, 'cv_auc': cv_auc}
    print(f'{name:<25} | ROC-AUC: {auc:.4f} | CV-AUC: {cv_auc:.4f} | Avg Prec: {ap:.4f}')

best_name = max(results, key=lambda k: results[k]['roc_auc'])
best      = results[best_name]
print(f'\nMelhor modelo: {best_name} (ROC-AUC = {best["roc_auc"]:.4f})')

In [ ]:
# Relatorio de Classificacao
print(f'Classification Report - {best_name}')
print(classification_report(y_test, best['y_pred'],
                             target_names=['Entregue (0)', 'Cancelado (1)']))

In [ ]:
# Visualizacoes do Modelo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'Avaliacao do Modelo - {best_name}',
             fontsize=14, fontweight='bold', color='white', y=1.01)
colors_roc = [C_CYAN, C_GREEN, C_YELLOW]

ax = axes[0, 0]
ax.set_facecolor('#161b22')
for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC={res['roc_auc']:.3f})")
ax.plot([0,1],[0,1], '--', color='#30363d', linewidth=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC', color='white')
ax.legend(fontsize=8, facecolor='#161b22', edgecolor='#30363d')
ax.grid(alpha=0.2)

ax = axes[0, 1]
ax.set_facecolor('#161b22')
for (name, res), color in zip(results.items(), colors_roc):
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    ax.plot(rec, prec, color=color, linewidth=2, label=f"{name} (AP={res['avg_precision']:.3f})")
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall', color='white')
ax.legend(fontsize=8, facecolor='#161b22', edgecolor='#30363d')
ax.grid(alpha=0.2)

ax = axes[1, 0]
ax.set_facecolor('#161b22')
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Entregue','Cancelado'],
            yticklabels=['Entregue','Cancelado'],
            ax=ax, cbar=False, annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Matriz de Confusao', color='white')
ax.set_xlabel('Predito'); ax.set_ylabel('Real')

ax = axes[1, 1]
ax.set_facecolor('#161b22')
mask0 = y_test == 0
mask1 = y_test == 1
ax.hist(best['y_proba'][mask0], bins=40, alpha=0.7, color=C_GREEN, label='Entregue (0)', density=True)
ax.hist(best['y_proba'][mask1], bins=40, alpha=0.7, color=C_RED, label='Cancelado (1)', density=True)
ax.axvline(0.5, color='white', linestyle='--', linewidth=1.5, label='Threshold 0.5')
ax.set_xlabel('Probabilidade Predita de Cancelamento')
ax.set_ylabel('Densidade')
ax.set_title('Distribuicao de Probabilidades', color='white')
ax.legend(fontsize=8, facecolor='#161b22', edgecolor='#30363d')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '09_model_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 9 salva')

In [ ]:
# Feature Importance
rf_pipe = pipelines['Random Forest']
rf_clf  = rf_pipe.named_steps['classifier']
prep    = rf_pipe.named_steps['preprocessor']
cat_encoder  = prep.named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(FEATURES_CAT).tolist()
all_features = FEATURES_NUM + cat_features
importances  = pd.Series(rf_clf.feature_importances_, index=all_features)
top_features = importances.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
colors_fi = [C_CYAN if i >= len(top_features) - 5 else C_BLUE for i in range(len(top_features))]
top_features.plot.barh(ax=ax, color=colors_fi, edgecolor='#0d1117')
ax.set_title('Feature Importance - Random Forest\n(Top 15 preditores de cancelamento)',
             fontsize=12, fontweight='bold', color='white', pad=12)
ax.set_xlabel('Importancia Relativa', fontsize=10)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '10_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 10 salva')
print('\nTop 5 preditores:')
print(importances.nlargest(5).to_string())

In [ ]:
# Simulacao Financeira
df_test = X_test.copy()
df_test['y_real']  = y_test.values
df_test['y_proba'] = best['y_proba']
df_test['y_pred']  = best['y_pred']

verdadeiros_cancel = df_test[(df_test['y_proba'] >= 0.5) & (df_test['y_real'] == 1)]
receita_em_risco   = verdadeiros_cancel['price'].sum()
receita_total_test = df_test[df_test['y_real'] == 1]['price'].sum()
pct_capturado      = receita_em_risco / receita_total_test * 100

print('=' * 55)
print(' SIMULACAO FINANCEIRA - IMPACTO DO MODELO')
print('=' * 55)
print(f'  Pedidos cancelados no teste       : {(df_test["y_real"]==1).sum():,}')
print(f'  Receita total em risco            : R$ {receita_total_test:,.2f}')
print(f'  Cancelamentos identificados       : {len(verdadeiros_cancel):,}')
print(f'  Receita potencialmente recuperavel: R$ {receita_em_risco:,.2f}')
print(f'  % da receita em risco capturada   : {pct_capturado:.1f}%')
print('=' * 55)
print('\nProjeto completo! Pipeline finalizado.')